<a href="https://colab.research.google.com/github/ashokmulchandani/CUDA-GPU-Colab-Computer-Vision-Project-Ashok-1/blob/main/2_Ashok_CUDA_LOVEACCOUNT_YOLO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Check GPU and setup
!nvidia-smi | head -5

Mon May 25 11:37:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |


In [ ]:
!pip install -q ultralytics
from ultralytics import YOLO
import time

# Download model + export to TensorRT (skip if already done)
import os
if not os.path.exists('yolov8n.engine'):
    model = YOLO('yolov8n.pt')
    model.export(format='engine', half=True)
    print("✓ TensorRT export done")
else:
    print("✓ TensorRT engine already exists")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 31.3 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
WARNING ⚠️ TensorRT requires GPU export, automatically assigning device=0
Ultralytics 8.4.53 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLOv8n summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs

PyTorch: starting from 'yolov8n.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 84, 8400) (6.2 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxslim>=0.1.71', 'onnxruntime-gpu'] not found,

In [ ]:
# Download test image
!wget -q https://ultralytics.com/images/bus.jpg -O test_image.jpg

# Benchmark
model_trt = YOLO('yolov8n.engine')
model_pt = YOLO('yolov8n.pt')

# Warmup both
for _ in range(5):
    model_trt('test_image.jpg', verbose=False)
    model_pt('test_image.jpg', verbose=False)

# Benchmark TensorRT
times = []
for _ in range(20):
    start = time.time()
    model_trt('test_image.jpg', verbose=False)
    times.append((time.time() - start) * 1000)
trt_avg = sum(times) / len(times)

# Benchmark PyTorch
times_pt = []
for _ in range(20):
    start = time.time()
    model_pt('test_image.jpg', verbose=False)
    times_pt.append((time.time() - start) * 1000)
pt_avg = sum(times_pt) / len(times_pt)

# Results
print(f"\n{'='*50}")
print(f"YOLO INFERENCE BENCHMARK")
print(f"{'='*50}")
print(f"PyTorch (FP32):    {pt_avg:.1f} ms  ({1000/pt_avg:.0f} FPS)")
print(f"TensorRT (FP16):   {trt_avg:.1f} ms  ({1000/trt_avg:.0f} FPS)")
print(f"Speedup:           {pt_avg/trt_avg:.1f}x faster!")
print(f"{'='*50}")
